# Bab 1: Common Conventions and API Elements of scikit-learn

Notebook ini merangkum Bab 1 buku *scikit-learn Cookbook* (Third Edition) karya John Sukup. Bab ini adalah fondasi sebelum masuk ke resep-resep yang lebih teknis: memahami filosofi desain scikit-learn dan elemen-elemen API yang dipakai berulang-ulang di seluruh buku, mulai dari estimator, transformer, pipeline, sampai metadata dan tuning hyperparameter.

Semua kode di notebook ini sudah dijalankan, jadi outputnya langsung terlihat begitu file dibuka.

## 1. Filosofi Desain scikit-learn

scikit-learn dibangun di atas empat prinsip inti: konsistensi, kesederhanaan, modularitas, dan reusability. Hampir semua model mengikuti pola yang sama: `fit()` untuk melatih, `predict()` untuk memprediksi, dan `transform()` untuk mengubah data. Begitu kamu paham polanya di satu model, kamu bisa langsung pindah ke model lain tanpa belajar API baru dari nol.

Satu catatan kecil yang sering bikin orang baru bingung: namanya memang ditulis huruf kecil semua, "scikit-learn", bukan "Scikit-Learn" atau "SciKit-Learn". Ini bukan typo, itu ejaan resmi dari pembuatnya. "Sci" adalah singkatan dari "science", jadi bisa dianggap sebagai "kit sains (data)".

## 2. Memahami Estimator

Estimator adalah objek yang mengimplementasikan algoritma pembelajaran dari data. Dua method paling penting: `fit()` untuk melatih model dari data training, dan `predict()` untuk membuat prediksi pada data baru.

In [1]:
from sklearn.linear_model import LinearRegression
import numpy as np

# Data contoh
X = np.array([[1], [2], [3], [4], [5]])  # Feature matrix
y = np.array([1, 2, 3, 3.5, 5])  # Target values

# Buat dan latih model
model = LinearRegression()
model.fit(X, y)

# Prediksi data baru
X_new = np.array([[6], [7]])
predictions = model.predict(X_new)
print(predictions)

[5.75 6.7 ]


Library ini juga menyediakan method pintas `fit_predict()` yang menggabungkan dua langkah sekaligus. Method ini biasanya dipakai dalam unsupervised learning, di mana kita ingin mendapat hasil prediksi pada dataset yang sama dengan yang dipakai untuk training, seperti contoh KMeans berikut.

In [2]:
from sklearn.cluster import KMeans

# Data contoh
X = np.array([[1], [2], [3], [4], [5]])

# Contoh clustering dengan KMeans
kmeans = KMeans(n_clusters=2, random_state=2024, n_init=10)
labels = kmeans.fit_predict(X)
print(labels)

[0 0 0 1 1]


Polanya tetap sama baik untuk model sesederhana regresi linear maupun model serumit random forest, itulah inti konsistensi scikit-learn.

## 3. Transformer dan Method transform()

Transformer memodifikasi data, misalnya scaling, normalisasi, atau encoding. Pola kerjanya: `fit()` untuk mempelajari parameter yang dibutuhkan dari data (misalnya mean dan standar deviasi), lalu `transform()` untuk menerapkan transformasi itu.

In [3]:
from sklearn.preprocessing import StandardScaler

# Data contoh
X = np.array([[1, 2], [3, 4], [5, 6]])

# Buat StandardScaler
scaler = StandardScaler()
scaler.fit(X)
X_scaled = scaler.transform(X)
print(X_scaled)

[[-1.22474487 -1.22474487]
 [ 0.          0.        ]
 [ 1.22474487  1.22474487]]


Ada juga method pintas `fit_transform()` yang menggabungkan `fit()` dan `transform()` dalam satu langkah. Ini cocok dipakai pada data training. Tapi untuk data testing, kita tidak boleh memanggil `fit()` lagi, cukup `transform()` saja memakai parameter yang sudah dipelajari dari data training. Kalau kita fit ulang di data testing, parameter transformasinya jadi berbeda dan bisa membuat prediksi model tidak bisa diandalkan saat dipakai di dunia nyata.

In [4]:
# fit_transform menggabungkan fit() dan transform() dalam satu langkah
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print(X_scaled)

[[-1.22474487 -1.22474487]
 [ 0.          0.        ]
 [ 1.22474487  1.22474487]]


## 4. Custom Estimator dan Transformer

API scikit-learn didesain agar mudah diperluas. Dengan subclassing `BaseEstimator()` dan kelas mixin, kita bisa membuat algoritma atau transformasi data sendiri yang tetap kompatibel dengan tools seperti `GridSearchCV()` dan `Pipeline()`.

Mixin adalah cara memperluas fungsi suatu class tanpa pakai inheritance tradisional, berguna untuk membagi fungsi yang sama ke banyak class tanpa menulis ulang kodenya. Estimator custom biasanya mengimplementasikan `fit()` dan `transform()` (untuk transformer) atau `fit()` dan `predict()` (untuk model).

In [5]:
from sklearn.base import BaseEstimator, TransformerMixin

class TambahKonstanta(BaseEstimator, TransformerMixin):
    """Contoh transformer custom sederhana: menambahkan konstanta ke semua nilai."""

    def __init__(self, konstanta=1.0):
        self.konstanta = konstanta

    def fit(self, X, y=None):
        # Tidak ada parameter yang perlu dipelajari dari data di contoh ini
        return self

    def transform(self, X):
        return np.asarray(X) + self.konstanta

# Uji coba transformer custom
custom_transformer = TambahKonstanta(konstanta=10)
X_contoh = np.array([[1, 2], [3, 4]])
print(custom_transformer.fit_transform(X_contoh))

[[11 12]
 [13 14]]


Karena mewarisi `BaseEstimator` dan `TransformerMixin`, transformer buatan sendiri ini otomatis punya method `fit_transform()`, dan bisa langsung dipakai di dalam `Pipeline()` seperti transformer bawaan scikit-learn lainnya.

## 5. Pipeline dan Otomatisasi Alur Kerja

`Pipeline()` memungkinkan kita merangkai beberapa langkah preprocessing dan training model jadi satu objek. Ini penting untuk MLOps: memastikan transformasi yang sama diterapkan secara konsisten di data training maupun data baru saat model sudah dipakai di produksi.

In [6]:
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

# Contoh pipeline sederhana: scaling lalu training model
pipeline_contoh = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", RandomForestClassifier(random_state=2024)),
])
pipeline_contoh

Pipeline(steps=[('scaler', StandardScaler()),
                ('classifier', RandomForestClassifier(random_state=2024))])

## 6. Atribut dan Method Umum

Model-model di scikit-learn berbagi sejumlah atribut dan method standar yang membantu kita memahami apa yang sudah dipelajari model. Contohnya `coef_` dan `intercept_` pada model linear, serta method `score()` untuk evaluasi performa cepat (R-squared untuk regressor, accuracy untuk classifier).

In [7]:
# Definisikan ulang data contoh supaya tidak tertukar dengan X dari contoh StandardScaler di atas
X = np.array([[1], [2], [3], [4], [5]])
y = np.array([1, 2, 3, 3.5, 5])

model = LinearRegression()
model.fit(X, y)

print("Coefficients:", model.coef_)
print("Intercept:", model.intercept_)
print("Model R-squared:", model.score(X, y))

Coefficients: [0.95]
Intercept: 0.04999999999999938
Model R-squared: 0.9809782608695652


R-squared sendiri punya keterbatasan: nilainya akan selalu naik begitu kita menambah variabel baru ke model, walau variabel itu tidak benar-benar relevan. Untuk itu, adjusted R-squared sering dipakai karena memberi penalti atas jumlah variabel yang dipakai.

## 7. Hyperparameter Tuning dengan Search Methods

Berbeda dengan parameter (yang dipelajari model saat training, seperti koefisien regresi), hyperparameter adalah pengaturan yang kita tentukan sebelum training untuk mengatur "cara belajar" model. scikit-learn menyediakan `GridSearchCV()` dan `RandomizedSearchCV()` untuk mencari kombinasi hyperparameter terbaik secara otomatis.

Selain itu, ada juga cara manual lewat `set_params()` untuk mengatur hyperparameter dan `get_params()` untuk melihat pengaturan yang sedang dipakai.

In [8]:
model = RandomForestClassifier()

# Atur hyperparameter secara manual
model.set_params(n_estimators=100, max_depth=10, random_state=42)

# Lihat semua parameter yang sedang dipakai
model.get_params()

{'bootstrap': True,
 'ccp_alpha': 0.0,
 'class_weight': None,
 'criterion': 'gini',
 'max_depth': 10,
 'max_features': 'sqrt',
 'max_leaf_nodes': None,
 'max_samples': None,
 'min_impurity_decrease': 0.0,
 'min_samples_leaf': 1,
 'min_samples_split': 2,
 'min_weight_fraction_leaf': 0.0,
 'monotonic_cst': None,
 'n_estimators': 100,
 'n_jobs': None,
 'oob_score': False,
 'random_state': 42,
 'verbose': 0,
 'warm_start': False}

## 8. Metadata: Tags dan Metadata Routing

scikit-learn menggunakan metadata, seperti estimator tags, untuk mengatur perilaku model dalam berbagai konteks, misalnya cross-validation dan pemrosesan di dalam pipeline. Tags ini memberi tahu scikit-learn kemampuan suatu estimator, misalnya apakah ia mendukung multi-output atau bisa menangani missing values.

Metadata routing memungkinkan informasi tambahan (seperti sample weights atau group labels) diteruskan ke komponen tertentu dalam pipeline, sambil tetap diabaikan oleh komponen lain yang tidak membutuhkannya. Ini berguna misalnya saat menangani dataset yang tidak seimbang, di mana sample weight perlu diteruskan ke classifier tapi tidak ke tahap scaling.

In [9]:
# Contoh melihat tags sebuah estimator
model = RandomForestClassifier()
tags = model.__sklearn_tags__()
print(tags)

Tags(estimator_type='classifier', target_tags=TargetTags(required=True, one_d_labels=False, two_d_labels=False, positive_only=False, multi_output=True, single_output=True), transformer_tags=None, classifier_tags=ClassifierTags(poor_score=False, multi_class=True, multi_label=True), regressor_tags=None, array_api_support=False, no_validation=False, non_deterministic=False, requires_fit=True, _skip_test=False, input_tags=InputTags(one_d_array=False, two_d_array=True, three_d_array=False, sparse=True, categorical=False, string=False, dict=False, positive_only=False, allow_nan=True, pairwise=False))


## 9. Best Practices Penggunaan API

Beberapa catatan penting yang berlaku di sepanjang buku ini:

- **API yang seragam**: semua estimator mengikuti pola `fit()`, `transform()`, dan `predict()` yang sama, membuat kode lebih mudah dibaca dan dirawat.
- **Preprocessing data**: selalu proses data lebih dulu lewat `sklearn.preprocessing` (scaling, encoding, menangani missing values) sebelum masuk ke model.
- **Pipeline**: untuk alur kerja yang melibatkan banyak transformasi dan model, gunakan `Pipeline()` supaya kode lebih sederhana dan tuning hyperparameter lebih mudah dikelola.
- **Cross-validation**: evaluasi performa model memakai teknik cross-validation dari `sklearn.model_selection` untuk mendapat estimasi kemampuan generalisasi yang lebih dapat diandalkan.
- **Hyperparameter tuning**: gunakan `GridSearchCV()` atau `RandomizedSearchCV()` untuk mencari hyperparameter optimal.

## Poin-poin Penting Bab 1

- scikit-learn dibangun di atas API yang konsisten: `fit()`, `predict()`, `transform()`, berlaku di hampir semua model dan transformer.
- `fit_predict()` dan `fit_transform()` adalah method pintas, tapi perlu dipakai dengan hati-hati, terutama soal kapan memanggil `fit()` ulang pada data testing (jangan!).
- Estimator dan transformer custom bisa dibuat dengan subclassing `BaseEstimator` dan mixin yang sesuai, supaya tetap kompatibel dengan `Pipeline()` dan `GridSearchCV()`.
- `Pipeline()` merangkai preprocessing dan modeling jadi satu objek yang konsisten dan mudah direproduksi, dasar penting untuk alur kerja MLOps.
- Atribut seperti `coef_`, `intercept_`, dan method seperti `score()` memberi cara cepat untuk memeriksa dan mengevaluasi model.
- Hyperparameter diatur sebelum training lewat `GridSearchCV()`, `RandomizedSearchCV()`, atau manual lewat `set_params()`/`get_params()`.
- Metadata dan tags membantu scikit-learn mengatur alur data secara otomatis di dalam pipeline, termasuk soal sample weight dan kompatibilitas tipe data.

Referensi: Sukup, J. (2025). *scikit-learn Cookbook*, Third Edition, Bab 1 "Common Conventions and API Elements of scikit-learn". Packt Publishing.